In [ ]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2
%matplotlib qt

In [ ]:
import os
os.chdir("..")


In [ ]:
from notebooks.experimental import *

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy
import napari

import osmnx as ox
ox.settings.use_cache = True
ox.settings.cache_folder = './osm_raw_cache'

from geo2stl import geo2stl as g2s
from geo2stl import sat2stl as s2s
from numpy2stl import numpy2stl as n2s
from city2stl import dem2stl as d2s
from city2stl import roads

from numpy2stl.numpy2stl import puzzle, view, verify, simplify, boolean
from numpy2stl.numpy2stl.save import writeOBJ, write3MF

In [ ]:
def load_data(Root, bbox, tags ):


	bounds_NW = ((bbox[3],bbox[1]),(bbox[0],bbox[2]))
	(N,S),(W,E) = bounds_NW


	if 1:
		print("retriving data")
		dem = d2s.DEM(Root, (N,S,E,W))
		im = dem.data.copy()
		sat = s2s.fetch_bbox_image(N*1., S*1., E*1., W*1., 
					scale=2, dataset="esa")

		print("Retriving Data from OSMNX")
		gdf = ox.features_from_bbox(bbox, tags)

		road_graph = ox.graph_from_bbox(bbox, network_type='drive')
		gdf_roads = ox.graph_to_gdfs(road_graph, nodes=False)

	return im, sat, gdf, gdf_roads

In [ ]:
def render_2D(gdf, gdf_roads, im, bbox):

	feat = "height"
	alpha = 0.5
	

	x = np.array(gdf[feat])
	x[0] = 0
	gdf[feat] = x 
	im[0,0] = gdf[feat].max()
	print(gdf[feat].min(), gdf[feat].max())

	fig, axs = plt.subplots(figsize=(15, 10), ncols=1, nrows=1, constrained_layout=True)# sharex=True, sharey=True, )
	axs = [axs]	
	
	
	axs[0].set_facecolor('#000000') # Dark background for a "pro" look

	# Plot Buildings colored by height


	gdf.plot(ax=axs[0], column=feat,cmap='jet', 
						alpha=alpha, edgecolor='#100', 
						linewidth=1, legend=True, zorder=2)
	if 0:
		gdf_roads.plot(
			ax=axs[0], 
			column=feat, 
			cmap='jet', 
			alpha=alpha, 
			edgecolor='#FFFFFF', 
			linewidth=0,
			zorder=2
		)

	axs[0].imshow(im, cmap="jet", extent=np.array(bbox)[[0,2,1,3]])	


In [ ]:
Root = "C:/Users/eac84/Desktop/Desktop/FILES"


In [ ]:
s2s.initialize_earth_engine()

TOLERANCE_M = 0.1            # Simplification tolerance in meters
ROAD_WIDTHS = {
	'motorway': 12, 'trunk': 10, 'primary': 8, 
	'secondary': 7, 'tertiary': 6, 'residential': 4, 'service': 2
}  

## Cities

In [ ]:
tags = {'building': True}

if 0:
	## Granada
	scale = .2
	center = (37.17827, -3.598) 
	dist = 1000

elif 1:
	name = "Philadelphia"
	scale = .1
	center = (39.952, -75.16356) 
	dist = 2500

elif 0:
	name = "Cartegena" 
	scale = .1
	center = (10.407, -75.545) 
	dist = 2500


In [ ]:
bbox = ox.utils_geo.bbox_from_point(center, dist)
bounds_NW = ((bbox[3],bbox[1]),(bbox[0],bbox[2]))
(N,S),(W,E) = bounds_NW

gdf_roads2 = None


im, sat, gdf, gdf_roads = load_data(Root, bbox, tags )


if 1:
	#im = dem.data.copy()
	for _ in range(3): im = scipy.ndimage.median_filter(im, 3)

	shape_out = im.shape
	outsize = np.array(shape_out)/max(shape_out)*100	
	im = transform.resize(im, outsize, anti_aliasing=True)	
	im = im - im.min() + im.ptp()*0.5

if 1:
	label = s2s.map_label_elevation(sat.copy() ,im, size=500)


if 1:

	if gdf is not None:
		gdf = fill_building_heights(gdf.copy())
		gdf = reduce_buildings(gdf, TOLERANCE_M=dist/1000, a=1)
		gdf_buildings = add_building_z(gdf, label, bounds_NW)

if 1:
	if gdf_roads is not None:

		gdf_roads2 = roads.get_road_segments(gdf_roads, ROAD_WIDTHS, TOLERANCE_M)
		polygon_list = polygon_list = list(gdf_roads2.geometry)
		gdf_roads2["z"] = roads.get_z_values(polygon_list, im, bounds_NW)
		gdf_roads2["z1"] = [x.mean() for x in gdf_roads2["z"]]	
		gdf_roads2["z0"] = 0

render_2D(gdf, gdf_roads2, label, bbox)

In [ ]:
im, sat, gdf, gdf_roads = load_data(Root, bbox, tags )

In [ ]:
out_dir = "C:/Users/eac84/OneDrive/Documents/Projects/3D Maps/Cities_/"

In [ ]:
import napari


models = {}
models["land"] = create.get_landspace_model(label, bounds_NW, scale, False)
models["buildings"] = create.get_building_model(gdf, scale*0.1)


#models["buildings2"] = get_building_model(gdf2, scale)
#models["terrain"] = get_landspace_model(label, bounds_NW, scale)
#models["space"] = get_bounds_model(gdf, scale)


In [ ]:
view.render_models_napari(models)

In [ ]:

out_dir2 = out_dir + "/" + name 
os.makedirs(out_dir2,exist_ok=True)
filename = out_dir2 + "/" + name + ".3mf"
write3MF(filename, models)

In [ ]:
gdf_buildings = fill_building_heights(gdf.copy())
gdf_buildings = reduce_buildings(gdf_buildings, TOLERANCE_M=dist/1000, a=1)
gdf_buildings = add_building_z(gdf_buildings, label*0, bounds_NW)
render_2D(gdf_buildings, gdf_roads2, label, bbox)

In [ ]:
plt.close("all")
render_2D(gdf, gdf_roads2, label)

In [ ]:
view.render_models_napari(models)

In [ ]:
for key in models:
	print("-"*20)
	print(key)

	vertices, faces = models[key]
	print(vertices.mean(axis=0))
	print(vertices.ptp(axis=0))

	mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
	is_watertight = mesh.is_watertight
	is_volume = mesh.is_volume 
	print(is_watertight , is_volume)


In [ ]:
plt.close("all")
plt.figure()

vertices, faces = models["landscape"]
surfaces, normals = n2s.get_surfaces(vertices[faces].copy(), normals=None)
vx, new_faces = simplify_mesh_surfaces(vertices.copy(),faces.copy(), surfaces.copy())
models["landscape2"] = (vertices,new_faces)

print("---- Simplification complete ------")
print(len(new_faces), len(faces), (len(new_faces)/len(faces)))
open_edges = n2s.get_open_edges(new_faces)
len(open_edges) 

mesh = trimesh.Trimesh(vertices=vertices, faces=new_faces)
is_watertight = mesh.is_watertight
is_volume = mesh.is_volume 
print(is_watertight , is_volume)



In [ ]:
for key in models:
	print(key)

	vertices, faces = models[key]

	print(vertices.mean(axis=0))
	print(vertices.min(axis=0), vertices.max(axis=0))

In [ ]:
%lprun -f reduce_surface simplify_mesh_surfaces(vertices.copy(),faces.copy(), surfaces.copy())